# Part 4 — SFTTrainer 학습 실행 (간소화 버전)
### 약 3~5분 안에 파인튜닝 체험하기

> **환경**: RTX 3080 + Part 3 완료 상태  
> **학습 규모**: 200건 × 2 에폭 = **약 50 스텝**  
> **소요 시간**: 3~5분

---

## 🎯 5단계

| # | 내용 |
|---|------|
| 1 | 데이터 로드 (200건) |
| 2 | 채팅 템플릿 적용 + 토크나이징 |
| 3 | TrainingArguments 설정 |
| 4 | Trainer.train() 실행 🚀 |
| 5 | 학습 후 응답 비교 + 저장 |

---

## ⚠️ 사전 조건
Part 3의 `model`, `tokenizer`, `CONFIG`, `baseline_results`, `TEST_QUESTIONS`, `generate_fast` 가 살아있어야 합니다.

---
## 1️⃣ 데이터 로드 (200건)

In [ ]:
from datasets import load_dataset
import random

ds = load_dataset("jihye-moon/LawQA-Ko", split="train")

# 컬럼 이름 확인
q_col, a_col = ds.column_names[:2]
print(f"컬럼: 질문='{q_col}', 답변='{a_col}'")

# 200건 샘플링
random.seed(42)
ds = ds.select(random.sample(range(len(ds)), 200))
print(f"✅ 학습 데이터: {len(ds)}건")

---
## 2️⃣ 포맷 + 토크나이징

In [ ]:
SYSTEM = "당신은 대한민국 법률 전문 AI 어시스턴트입니다."

def preprocess(sample):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": sample[q_col]},
        {"role": "assistant", "content": sample[a_col]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tok = tokenizer(text, truncation=True, max_length=1024, padding=False)
    tok["labels"] = tok["input_ids"].copy()
    return tok

tokenized = ds.map(preprocess, remove_columns=ds.column_names)
print(f"✅ 토크나이징 완료: {len(tokenized)}건")

---
## 3️⃣ TrainingArguments (50스텝 설정)

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir="./exaone4-law-quick",
    num_train_epochs=2,                        # 2 에폭
    per_device_train_batch_size=2,             # 배치 2
    gradient_accumulation_steps=4,             # 실효 배치 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="no",                        # 체크포인트 저장 안함 (속도)
    report_to="none",
    seed=42,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8)
예상스텝 = len(tokenized) // 8 * 2
print(f"✅ 예상 스텝: 약 {예상스텝}")

---
## 4️⃣ 학습 실행 🚀 (3~5분)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)

print("🏋️ 학습 시작!\n")
result = trainer.train()

print(f"\n✅ 학습 완료!")
print(f"   train_loss: {result.training_loss:.4f}")
print(f"   소요 시간:  {result.metrics['train_runtime']:.0f}초")

---
## 5️⃣ 학습 후 응답 + 저장

In [ ]:
model.eval()

# 첫 질문으로 Before/After 비교
질문 = TEST_QUESTIONS[0]
학습후 = generate_fast(model, tokenizer, 질문)
베이스라인 = baseline_results[0]["baseline_answer"]

print(f"[질문] {질문}\n")
print("=" * 55)
print("[학습 전]")
print("=" * 55)
print(베이스라인[:300])
print("\n" + "=" * 55)
print("[학습 후]")
print("=" * 55)
print(학습후[:300])

In [ ]:
# 5개 질문 전체 수집
finetuned_results = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"[{i}/5] 생성 중...")
    finetuned_results.append({
        "question": q,
        "baseline": baseline_results[i-1]["baseline_answer"],
        "finetuned_raw": generate_fast(model, tokenizer, q),
    })

print(f"\n✅ {len(finetuned_results)}건 완료")

In [ ]:
# 어댑터 + 결과 저장
import json, os

trainer.save_model("./exaone4-law-raw")
tokenizer.save_pretrained("./exaone4-law-raw")

os.makedirs("./outputs", exist_ok=True)
with open("./outputs/finetuned_raw_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "train_loss": result.training_loss,
        "results": finetuned_results,
    }, f, ensure_ascii=False, indent=2)

print("✅ 어댑터 저장: ./exaone4-law-raw/")
print("✅ 결과 저장:   ./outputs/finetuned_raw_results.json")

---
## 🎓 Part 4 완료

| 결과물 | 경로 |
|---|---|
| LoRA 어댑터 | `./exaone4-law-raw/` |
| 학습 결과 JSON | `./outputs/finetuned_raw_results.json` |

### 🔜 Part 5 예고
GPT-4o-mini로 데이터 정제 → 재학습 → **원본 학습 vs 정제 학습 비교**